# Create MinCiencias Awards (GRANT PATTERN, Colombia)

Colombian research and innovation projects from datos.gov.co (the
Colombian national open-data portal), Socrata dataset `6hgx-q9pi`:
"Proyectos de Investigación e Innovación evaluados y aprobados desde
el año 2009". 3,140 projects after dedup (2007-2021).

**Awarding bodies (year-bounded, dual funder_id):**
- 2019-onwards: Ministerio de Ciencia, Tecnología e Innovación (`F3277441329`)
- pre-2019:    COLCIENCIAS / Departamento Administrativo de Ciencia,
               Tecnología e Innovación (`F4320309955`) — the legal
               predecessor (MinCiencias was created in 2019 as a cabinet
               ministry; before that the same agency operated as an
               administrative department under the name Colciencias).

The `funder_id` is selected per-row by `ano_convocatoria` in the Step 2
transform, analogous to Abel Prize's NOK 6M/7.5M boundary in 2019.

**Schema notes (grant pattern, monetary):**
- `amount` = `monto_financiado_ap` (the funder's share — MinCiencias's
  contribution). `monto_contrapartida_ap` (counterpart funding from the
  recipient institution) and `monto_total_ap` (sum of both) are present
  in the raw table for downstream reference but not used as `amount`.
- `currency` = `'COP'` (Colombian Peso). The source has no currency
  column — implicit per the runbook's hardcode-from-region rule (single
  national funder).
- `lead_investigator.given_name` / `family_name` / `orcid` are **NULL by
  source-authority** — the dataset does not publish PI data. We do
  populate `affiliation.name` from `entidad_ejecuta` and
  `affiliation.country = 'CO'`.
- ~91% have a non-zero amount; ~9% are zeros (likely projects approved
  in principle but not yet funded, or canceled). The §6.7 check uses
  the standard >50% pct_amount threshold and passes.

**Prerequisites:**
- Run `scripts/local/minciencias_to_s3.py` first to fetch the Socrata
  dataset and upload the parquet to S3.

**Data source:** https://www.datos.gov.co/d/6hgx-q9pi
**S3 location:** `s3a://openalex-ingest/awards/minciencias/minciencias_projects.parquet`


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.minciencias_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/minciencias/minciencias_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.minciencias_raw;

## Step 1.5: Inspect raw + money-field scan

**Critical step for grant pattern.** The runbook calls out DataCite and
GTR as past ingests that shipped with NULL amount/currency because the
money-field search was too narrow. Verify the column names exist as
expected and that the amount distribution is plausible (avg in
thousands-to-millions, not 1-100 or 1900-2030).

In [ ]:
%sql
DESCRIBE openalex.awards.minciencias_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.minciencias_raw LIMIT 5;

In [ ]:
%sql
-- Money-field scan: identify every column with money-shaped name (case-insensitive)
SELECT column_name FROM (DESCRIBE openalex.awards.minciencias_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';

In [ ]:
%sql
-- Sanity-check the candidate amount column distribution.
-- Real grant amounts in COP sit in the hundreds-of-thousands to billions.
SELECT
    MIN(TRY_CAST(monto_financiado_ap AS DOUBLE)) AS min_val,
    MAX(TRY_CAST(monto_financiado_ap AS DOUBLE)) AS max_val,
    AVG(TRY_CAST(monto_financiado_ap AS DOUBLE)) AS avg_val,
    COUNT(monto_financiado_ap) AS non_null,
    SUM(CASE WHEN monto_financiado_ap > 0 THEN 1 ELSE 0 END) AS non_zero,
    COUNT(*) AS total_rows
FROM openalex.awards.minciencias_raw;

In [ ]:
%sql
-- General coverage check
SELECT
    COUNT(*) AS total_rows,
    COUNT(titulo_proyecto) AS has_title,
    COUNT(ano_convocatoria) AS has_year,
    COUNT(entidad_ejecuta) AS has_institution,
    COUNT(fecha_aprobacion) AS has_approval_date,
    COUNT(slug) AS has_slug,
    MIN(ano_convocatoria) AS min_year,
    MAX(ano_convocatoria) AS max_year,
    COUNT(DISTINCT ano_convocatoria) AS distinct_years,
    SUM(CASE WHEN ano_convocatoria < 2019 THEN 1 ELSE 0 END) AS colciencias_era,
    SUM(CASE WHEN ano_convocatoria >= 2019 THEN 1 ELSE 0 END) AS minciencias_era
FROM openalex.awards.minciencias_raw;

## Step 1.6: Fail-fast — verify BOTH funder rows exist

The transform in Step 2 joins each row against the appropriate funder
in `openalex.common.funder` (year-bounded — Colciencias for pre-2019,
MinCiencias for 2019+). If either funder row is absent, the join silently
emits zero rows for that era. Assert presence of both before transforming.

In [ ]:
%sql
-- Must return exactly 2 rows.
-- F3277441329 = Ministerio de Ciencia, Tecnología e Innovación (2019+)
-- F4320309955 = COLCIENCIAS (pre-2019)
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id IN (3277441329, 4320309955)
ORDER BY funder_id;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.minciencias_awards
USING delta
AS
WITH funders_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id IN (3277441329, 4320309955)
),
raw_with_funder AS (
    -- Year-bounded funder_id pick (Colciencias renamed to MinCiencias in 2019)
    SELECT
        n.*,
        CASE
            WHEN TRY_CAST(n.ano_convocatoria AS INT) >= 2019 THEN 3277441329
            WHEN TRY_CAST(n.ano_convocatoria AS INT) <  2019 THEN 4320309955
            ELSE NULL
        END AS picked_funder_id
    FROM openalex.awards.minciencias_raw n
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(rwf.picked_funder_id AS STRING), ':minciencias:', rwf.slug
    ))) % 9000000000 AS id,
    rwf.titulo_proyecto AS display_name,
    rwf.desc_convocatoria AS description,
    rwf.picked_funder_id AS funder_id,
    CONCAT('minciencias-', rwf.proyecto_id) AS funder_award_id,
    -- Amount: MinCiencias's share (financiado_ap). Counterpart funding
    -- (contrapartida_ap) and total (total_ap) are in the raw table for
    -- reference but not used as the canonical award amount.
    TRY_CAST(rwf.monto_financiado_ap AS DOUBLE) AS amount,
    'COP' AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    rwf.nme_prog_cti AS funder_scheme,
    'datos_gov_co_minciencias' AS provenance,
    -- Project approval date as canonical start_date; year boundary already enforced upstream.
    TRY_TO_DATE(SUBSTRING(rwf.fecha_aprobacion, 1, 10), 'yyyy-MM-dd') AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(rwf.ano_convocatoria AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    -- No PI fields in the source. Affiliation populated from entidad_ejecuta.
    struct(
        CAST(NULL AS STRING) AS given_name,
        CAST(NULL AS STRING) AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            rwf.entidad_ejecuta AS name,
            'CO' AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rwf.source_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(rwf.picked_funder_id AS STRING), ':minciencias:', rwf.slug
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_with_funder rwf
JOIN funders_resolved f
  ON rwf.picked_funder_id = f.funder_id
WHERE rwf.slug IS NOT NULL
  AND rwf.ano_convocatoria IS NOT NULL
  AND rwf.picked_funder_id IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 52

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'datos_gov_co_minciencias' AND priority = 52;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    52 AS priority
FROM openalex.awards.minciencias_awards;

## Step 6: Verification

Grant pattern, monetary → §6.7 fail-fast applies in full (not waived).
Expected: ~3,140 rows, pct_amount ~91% (the 9% zero-amount rows survive
the transform because they're real projects, just not yet funded /
canceled per the source). Currency = 'COP'. Year range 2007-2021.

In [ ]:
%sql
SELECT COUNT(*) AS total_minciencias_award_rows FROM openalex.awards.minciencias_awards;

In [ ]:
%sql
-- §6.2 Schema validation on the transformed table
DESCRIBE openalex.awards.minciencias_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook canonical query)
-- NOTE: pct_amount expected ~91% (9% zero-amount projects are kept;
-- they are real grants that were approved but not yet funded per source).
-- pct_dates expected 100%. has_pi expected 0 (no PI fields in source);
-- the lead_investigator struct is populated with affiliation only.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount_nonnull,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi_struct,
    SUM(CASE WHEN lead_investigator.given_name IS NOT NULL THEN 1 ELSE 0 END) AS has_pi_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_pi_affiliation,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.minciencias_awards;

In [ ]:
%sql
-- §6.7 amount/currency fail-fast check (NOT waived — monetary grant pattern)
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_nonzero_amt
FROM openalex.awards.minciencias_awards;

In [ ]:
%sql
-- Sample inspection
SELECT id, SUBSTRING(display_name, 1, 60) AS title,
       start_year, ROUND(amount, 0) AS amount, currency, funder_award_id,
       SUBSTRING(lead_investigator.affiliation.name, 1, 40) AS institution,
       funder.display_name AS funder_name
FROM openalex.awards.minciencias_awards
ORDER BY start_year DESC, id
LIMIT 12;

In [ ]:
%sql
-- Year distribution
SELECT start_year, COUNT(*) AS projects
FROM openalex.awards.minciencias_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Dual-funder split (sanity-check the 2019 boundary fired correctly)
SELECT funder.id, funder.display_name, COUNT(*) AS rows,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year
FROM openalex.awards.minciencias_awards
GROUP BY funder.id, funder.display_name
ORDER BY funder.display_name;